In [ ]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd  # A principal biblioteca de Data Science para tabelas (DataFrames) e manipulação matricial.
import pandas_gbq

# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [ ]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE AUTENTICAÇÃO E CONEXÃO COM O BIGQUERY
# ==============================================================================

# Caminho local do arquivo JSON contendo as credenciais privadas da conta de serviço.
# ATENÇÃO: Nunca versionar este arquivo no GitHub público.
meu_service_account_file = "../secrets/gcp-sa-seminarios.json"

# ID do projeto pessoal criado no Google Cloud Platform.
meu_gcp_project_id = "meu-projeto-eng-dados"  # ID do seu projeto pessoal criado no GCP.

# Conversão do arquivo JSON em um objeto de credenciais autenticadas.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# Inicialização do cliente de comunicação com o BigQuery.
# Este objeto será utilizado para leitura e escrita de tabelas na nuvem.
bigquery_client = bigquery.Client(credentials=gcp_credentials, project=meu_gcp_project_id)

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados financeiros e comerciais de filmes da Marvel e DC.
#
# Dataset:
#   bronze.boxoffice_dc_marvel
# ------------------------------------------------------------------------------

df_boxoffice_dc_marvel = bigquery_client.query("""
    SELECT *
    FROM bronze.boxoffice_dc_marvel
    """).to_dataframe(create_bqstorage_client=False)

In [ ]:
df_imdb_filmes = bigquery_client.query(
    "SELECT * FROM bronze.imdb_filmes"
).to_dataframe(create_bqstorage_client=False)

In [ ]:
df_complete_movie_info = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],
        right_on=["TITLE", "YEAR"],
        how="outer",
    )
    .sort_values(by="YEAR", ascending=False)
    .drop_duplicates(keep="last")
)

In [ ]:
df_complete_dc_marvel = df_complete_movie_info[
    (df_complete_movie_info["FRANCHISE"] == "Marvel")
    | (df_complete_movie_info["FRANCHISE"] == "DC")
]

In [ ]:
# ==============================================================================
# 7. REMOÇÃO DE COLUNAS REDUNDANTES E DESNECESSÁRIAS
# ==============================================================================

# Eliminando colunas:
#   - duplicadas após o merge,
#   - auxiliares,
#   - pouco relevantes para análises finais,
#   - ou já representadas por outras métricas equivalentes.

df_complete_dc_marvel = df_complete_dc_marvel.drop(
    columns=[
        "25X_PROD",
        "BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA",
        "BOX_OFFICE_GROSS_WORLDWIDE",
        "BUDGET_y",
        "DURATION",
        "LENGTH",
        "MPAA_RATING",
        "TITLE",
        "YEAR",
    ]
).reset_index(drop=True)

In [ ]:
df_complete_dc_marvel.info()

In [ ]:
df_complete_dc_marvel.sort_values("RATING_IMDB", ascending=False).head(5)

In [ ]:
df_tratado = df_complete_dc_marvel.copy()

In [ ]:
money_cols = [
    "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
    "BUDGET_x",
    "INFLATION_ADJUSTED_BUDGET",
    "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
    "GROSS_OPENING_WEEKEND",
    "GROSS_US_CANADA",
    "GROSS_WORLD_WIDE",
]

for col in money_cols:    
    df_tratado[col] = (
        df_tratado[col].astype(str).str.replace("$", "").str.replace(",", "").replace("None", 0)
        .astype(float)
    )


In [ ]:
percent_cols = ["DOMESTIC"]

for col in percent_cols:
    df_tratado[col] = df_tratado[col].astype(str).str.replace("%", "").astype(float)

In [ ]:
date_cols = ["US_RELEASE_DATE"]

for col in date_cols:
    df_tratado[col] = pd.to_datetime(df_tratado[col], format="%d/%m/%Y")

In [ ]:
int_cols = [
    "VOTE",
    "WIN",
    "OSCAR",
    "NOMINATION",
    "ROTTEN_TOMATOES_CRITIC_SCORE",
]

for col in int_cols:
    df_tratado[col] = df_tratado[col].fillna(0).astype(int)

In [ ]:
df_tratado["YEAR"] = df_tratado["US_RELEASE_DATE"].dt.year

In [ ]:
df_tratado["GENRE"]

In [ ]:
df_tratado["GENRE"] = df_tratado["GENRE"].str.split(",").str[0].str.strip()

df_tratado["COUNTRY_ORIGIN"] = df_tratado["COUNTRY_ORIGIN"].str.split(",").str[0].str.strip()

df_tratado["LANGUAGE"] = df_tratado["LANGUAGE"].str.split(",").str[0].str.strip()

df_tratado["PRODUCTION_COMPANY"] = df_tratado["PRODUCTION_COMPANY"].str.split(",").str[0].str.strip()

df_tratado["WRITER"] = df_tratado["WRITER"].str.split(",").str[0].str.strip()

In [ ]:
df_tratado.head()

In [ ]:
# df_tratado = df_tratado.drop(columns=["int64_field_0"])
df_tratado = df_tratado.drop(columns=["Unnamed: 0"])

In [ ]:
df_tratado = df_tratado[sorted(df_tratado.columns)].drop_duplicates()

In [ ]:
pandas_gbq.to_gbq(
    df_tratado,
    project_id=meu_gcp_project_id,
    destination_table="silver.complete_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)